## **Refine-Exp-Library-With-GPF**

Here, use the results from the GPF (formatted into a library) in order to refine the prediction models. This will then be used to create a new library with the same number of peptide precursors as the original library however refined IM/RT/Intensity.

In [4]:
from alphabase.peptide.fragment import get_charged_frag_types
import pandas as pd
from peptdeep.pretrained_models import ModelManager
import re
from alphabase.spectral_library.reader import LibraryReaderBase
from peptdeep.spec_lib.predict_lib import PredictSpecLib
from alphabase.spectral_library.reader import SWATHLibraryReader

In [5]:
def modified_sequence_to_tuple(seq):
    out = modified_sequence_to_tuple_helper(seq, '', '')
    return out[0] + out[1] + out[2]

def modified_sequence_to_tuple_helper(seq, mods, pos, translate_mod_dict = {'(UniMod:4)':'Carbamidomethyl@C', 
                                                                            '(UniMod:1)':'Acetyl@Protein N-term',
                                                                            '(UniMod:35)':'Oxidation@M', 
                                                                            '(UniMod:27)':'Glu->pyro-Glu@E^Any N-term', 
                                                                            '(UniMod:28)':'Gln->pyro-Glu@Q^Any N-term', 
                                                                            '(UniMod:26)':'Pyro-carbamidomethyl@C^Any N-term' 
                                                                           }):
    searchRslt =  re.search(r'\(UniMod:\d{1,2}\)', seq)
    if searchRslt == None:
        return (seq, mods, pos)
    
    else:
        unimod = seq[searchRslt.span()[0]:searchRslt.span()[1]]
        if mods != '':
            mods += ';'
            pos += ';'
        mods += translate_mod_dict[unimod]
        pos += str(searchRslt.span()[0])
        seq = seq[:searchRslt.span()[0]] + seq[searchRslt.span()[1]:]
        return modified_sequence_to_tuple_helper(seq, mods, pos, translate_mod_dict)

In [6]:
model_mgr = ModelManager(device='gpu')
model_mgr.nce = 30
model_mgr.instrument = "timsTOF"
frag_types = get_charged_frag_types(['b', 'y'], 2) 

/banana/rostlab/jcharkow/.conda/envs/peptdeep/lib/python3.9/site-packages/peptdeep/model/model_interface.py:731: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(str

### **Format Library for DIA-NN Input**

In [12]:
lib.columns

Index(['Unnamed: 0', 'PrecursorMz', 'ProductMz', 'ProteinId',
       'ModifiedPeptideSequence', 'PeptideSequence', 'PrecursorCharge',
       'PeptideGroupLabel', 'NormalizedRetentionTime', 'PrecursorIonMobility',
       'LibraryIntensity', 'Annotation', 'Precursor'],
      dtype='object')

In [9]:
lib = pd.read_csv("osw/2025-03-07-OSW-GPF-Lib.tsv", sep='\t')

In [22]:
lib['ModifiedPeptideSequence'] = lib['ModifiedPeptideSequence'].str.replace('.','', regex=False)
lib['FragmentType'] = lib['Annotation'].str.slice(start=0, stop=1)
lib['FragmentCharge'] = lib['Annotation'].str.slice(start=-1)
lib['FragmentSeriesNumber'] = lib['Annotation'].str.extract(r'[yb](\d+)\^')

lib.to_csv("osw/2025-03-07-OSW-GPF-Lib-Formatted-AlphaPeptDeep.tsv", index=False, sep='\t')

### **Load Library**

In [23]:
lib_obj = LibraryReaderBase()
lib = lib_obj.import_file("osw/2025-03-07-OSW-GPF-Lib-Formatted-AlphaPeptDeep.tsv")

/banana/rostlab/jcharkow/.conda/envs/peptdeep/lib/python3.9/site-packages/alphabase/psm_reader/psm_reader.py:168: FutureWarning: Passed unknown arguments to LibraryReaderBase (fixed_C57=False) will be forbidden in alphabase>1.5.0.
  warnings.warn(
100%|█████████████████████████████████| 116240/116240 [01:08<00:00, 1695.51it/s]


In [24]:
lib

,charge,mobility,mod_sites,mods,nAA,precursor_mz,proteins,rt,sequence,frag_start_idx,frag_stop_idx,rt_norm,ccs
0,1,1.004792,,,7,733.3839,P07910,16.368970,ASNVTNK,0,6,0.172373,205.014425
1,1,1.044976,,,7,757.4091,Q8WUM4,43.514211,DLDPIGK,6,12,0.458226,213.089007
2,1,1.073985,,,7,713.4920,P26640,40.435175,AVLVALK,12,18,0.425802,219.244709
3,1,1.079247,,,8,858.4931,Q12906,32.042929,PTTALLDK,18,25,0.337428,219.615044
4,1,1.094526,,,7,717.3778,P31948,15.519533,PSDLGTK,25,31,0.163428,223.415104
...,...,...,...,...,...,...,...,...,...,...,...,...,...
116234,5,1.023577,3;5;22;27,Carbamidomethyl@C;Carbamidomethyl@C;Carbamidom...,30,707.7388,Q9Y6R0,82.661767,WICHCFLALKDSGERLSHAVGCAFAACLER,1535122,1535151,0.870469,1028.902367
116235,5,1.024175,13;18,Carbamidomethyl@C;Carbamidomethyl@C,30,753.7600,P22314,74.875848,SLVLQRPQTWADCVTWACHHWHTQYSNNIR,1535151,1535180,0.788479,1029.256060
116236,5,1.025380,,,30,679.3550,Q92499,78.602060,IMHFPTWVDLKGEDSVPDTVHHVVVPVNPK,1535180,1535209,0.827718,1030.883776
116237,5,1.037122,,,30,723.1267,P27986,66.936594,KLNEWLGNENTEDQYSLVEDDEDLPHHDEK,1535209,1535238,0.704875,1042.431046


In [25]:
frag_intens = lib_obj.fragment_intensity_df

### **Transfer Learning**

In [26]:
help(model_mgr.train_ccs_model)

Help on method train_ccs_model in module peptdeep.pretrained_models:

train_ccs_model(psm_df: pandas.core.frame.DataFrame) method of peptdeep.pretrained_models.ModelManager instance
    Train/fine-tune the CCS model. The fine-tuning will be skipped
    if `self.psm_num_to_train_rt_ccs` is zero.
    
    Parameters
    ----------
    psm_df : pd.DataFrame
        Training psm_df which contains 'ccs' or 'mobility' column.



In [29]:
model_mgr.train_ccs_model(lib)

2025-06-02 15:24:50> 116236 PSMs for CCS model training/transfer learning


In [30]:
model_mgr.train_rt_model(lib)

2025-06-02 15:28:20> 97399 PSMs for RT model training/transfer learning


In [31]:
model_mgr.train_ms2_model(lib, frag_intens)

2025-06-02 15:30:38> 116239 PSMs for MS2 model training/transfer learning
2025-06-02 15:34:24> Testing refined MS2 model on training df:
            PCC       COS        SA            SPC
count  116239.0  116239.0  116239.0  116239.000000
mean        0.0       0.0       0.0       0.652931
std         0.0       0.0       0.0       0.115552
min         0.0       0.0       0.0       0.354158
25%         0.0       0.0       0.0       0.588244
50%         0.0       0.0       0.0       0.676630
75%         0.0       0.0       0.0       0.733854
max         0.0       0.0       0.0       0.973452
>0.90       0.0       0.0       0.0       0.001600
>0.75       0.0       0.0       0.0       0.202798


In [32]:
model_mgr.ccs_model.save("osw_models/ccs_model/ccs_model_OSW_GPF_refined.pth")

In [33]:
model_mgr.rt_model.save("osw_models/rt_model/rt_model_OSW_GPF_refined.pth")

In [34]:
model_mgr.ms2_model.save("osw_models/intens_model/intens_model_OSW_GPF_refined.pth")

#### **Produce the New Library**

##### **Load the Experimental Library**

In [46]:
expLib = LibraryReaderBase()
expLib.import_file("../K562-Library-Generation/library.tsv")

/banana/rostlab/jcharkow/.conda/envs/peptdeep/lib/python3.9/site-packages/alphabase/psm_reader/psm_reader.py:168: FutureWarning: Passed unknown arguments to LibraryReaderBase (fixed_C57=False) will be forbidden in alphabase>1.5.0.
  warnings.warn(
100%|█████████████████████████████████| 148389/148389 [01:39<00:00, 1488.26it/s]


,charge,genes,mobility,mod_sites,mods,nAA,precursor_mz,proteins,rt,sequence,frag_start_idx,frag_stop_idx,rt_norm,ccs
0,1,AAMP,1.686728,,,13,1320.627774,Q13685,32.576085,TNTLAVTGGEDDK,0,12,0.328517,341.328812
1,1,AARS1,1.592983,0,Acetyl@Any_N-term,11,1265.604201,P49588,76.463234,MDSTLTASEIR,12,22,0.764676,322.503879
2,1,ABCE1,1.401734,,,9,1077.452374,P61221,92.877751,SGNYFFLDD,22,30,0.927807,284.320836
3,1,ABCE1,1.614719,,,12,1316.644092,P61221,46.293520,NVEDLSGGELQR,30,41,0.464844,326.767241
4,1,ACADM,1.616148,,,12,1376.694397,P11310,72.345846,EEIIPVAAEYDK,41,52,0.723757,326.907980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148384,5,ZNF598,0.869284,2;5,Carbamidomethyl@C;Carbamidomethyl@C,24,523.653053,Q86UK7,34.661524,VCPTCQQVLAHGDASSHQALHAAR,1928847,1928870,0.349243,875.010883
148385,5,ZNF689,0.867406,17;20,Carbamidomethyl@C;Carbamidomethyl@C,22,527.447743,Q96CS4,21.793201,NLSQHQVIHTGEKPYHCPDCGR,1928870,1928891,0.221355,873.088224
148386,5,ZW10,0.750335,,,15,374.198816,O43264,78.587715,NIFHLFHDVVPTYHK,1928891,1928905,0.785790,756.872882
148387,5,ZW10,0.966057,12,Carbamidomethyl@C,27,641.934042,O43264,77.349905,LPQLAAIHHNNCMYIAHHLLTLGHQFR,1928905,1928931,0.773488,971.473184


In [47]:
libOut = PredictSpecLib()
expLib.precursor_df['instrument'] = model_mgr.instrument
expLib.precursor_df['nce'] = model_mgr.nce
libOut.model_manager = model_mgr
res = libOut.model_manager.predict_all(
    expLib.precursor_df,
    predict_items=['rt','mobility','ms2'],
    frag_types = frag_types,
    )
libOut.set_precursor_and_fragment(
    **res
    )

libOut.translate_rt_to_irt_pred()


2025-06-02 15:50:39> Predicting RT ...


100%|███████████████████████████████████████████| 24/24 [00:02<00:00,  8.44it/s]

2025-06-02 15:50:42> Predicting mobility ...



100%|███████████████████████████████████████████| 24/24 [00:03<00:00,  6.66it/s]


2025-06-02 15:50:47> Predicting MS2 ...


100%|███████████████████████████████████████████| 24/24 [00:05<00:00,  4.79it/s]


Predict RT for 11 iRT precursors.
Linear regression of `rt_pred` to `irt`:
   R_square         R      slope  intercept  test_num
0  0.995018  0.997506  160.37112 -57.592161        11


,charge,genes,mobility,mod_sites,mods,nAA,precursor_mz,proteins,rt,sequence,...,ccs,instrument,nce,rt_pred,rt_norm_pred,ccs_pred,mobility_pred,frag_start_idx,frag_stop_idx,irt_pred
0,2,RPS2,0.800479,,,7,440.724846,P15880,16.489480,RGYWGNK,...,325.643813,timsTOF,30,0.218414,0.218414,323.016998,0.794022,0,6,-22.564853
1,2,STXBP2,0.783983,5;6,Carbamidomethyl@C;Carbamidomethyl@C,7,434.206783,Q15833,13.728746,ILSSCCK,...,319.006798,timsTOF,30,0.195454,0.195454,313.473999,0.770386,6,12,-26.246908
2,2,STXBP2,0.807887,,,7,461.744390,Q15833,10.631236,KMPQYQK,...,328.426891,timsTOF,30,0.193456,0.193456,330.958221,0.814114,12,18,-26.567412
3,3,NUDT4,0.684781,,,7,297.499611,Q9NZJ9,6.537409,FKPNQTR,...,417.785192,timsTOF,30,0.159528,0.159528,399.638062,0.655037,18,24,-32.008520
4,2,STXBP3,0.766617,,,7,400.747456,O00186,35.038751,LAQLVEK,...,312.346762,timsTOF,30,0.321872,0.321872,305.448151,0.749685,24,30,-5.973237
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148384,3,USP28,1.063281,,,30,1061.809226,Q96RU2,46.344838,IPQMESSTNSSSQDYSTSQEPSVASSHGVR,...,641.566448,timsTOF,30,0.446318,0.446318,664.458862,1.101221,1928808,1928837,13.984294
148385,3,RRBP1,1.290266,,,30,987.227826,Q9P2E9,95.203877,APAVAVAPTPVQPPIIVAPVATVPAMPQEK,...,778.782214,timsTOF,30,0.930994,0.930994,744.296326,1.233131,1928837,1928866,91.712357
148386,3,ARHGAP45,1.055783,23,Carbamidomethyl@C,30,1002.124485,Q92619,57.552746,SSFNVSDVARPEAAGSPPEEGGCTEGTPAK,...,637.207679,timsTOF,30,0.538611,0.538611,629.629944,1.043227,1928866,1928895,28.785411
148387,5,DDX1,1.044447,,,30,679.355023,Q92499,83.093003,IMHFPTWVDLKGEDSVPDTVHHVVVPVNPK,...,1050.052536,timsTOF,30,0.834038,0.834038,991.267639,0.985976,1928895,1928924,76.163469


In [48]:
libOut.save_hdf("osw/2025-06-02-Exp-Lib-Transfer-Learn-GPF-Models-OSW.hdf")